In [1]:
!pip install langchain
!pip install openai
!pip install PyPDF2
!pip install faiss-cpu
!pip install tiktoken
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

In [ ]:
from PyPDF2 import PdfReader
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS


In [2]:
import os
os.environ["OPENAI_API_KEY"]= ""


In [ ]:
#provide pdf path
pdfreader=  PdfReader("/content/A_Step_Towards_Automated_Tool_Tracking_on_Construction_Sites_Boston_Dynamics_SPOT_and_RFID.pdf")

In [ ]:
from typing_extensions import Concatenate
#read text
raw_text= " "
for i, page in enumerate(pdfreader.pages):
  content= page.extract_text()
  if content:
    raw_text += content

In [ ]:
#splitting text to ensure token size
text_splitter= CharacterTextSplitter(
    separator="\n",
    chunk_size=800,
    chunk_overlap=200,
    length_function=len
)
texts= text_splitter.split_text(raw_text)

In [ ]:
#download embeddings from OpenAI
embeddings= OpenAIEmbeddings()

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  warn_deprecated(


In [ ]:
!pip install faiss-cpu
from langchain.vectorstores import FAISS
import time

document_search = []
for i in range(0, len(texts), 10):  # Process 10 chunks at a time
    chunk = texts[i : i + 10]
    # Create a FAISS object from the chunk and append it to the list
    document_search.append(FAISS.from_texts(chunk, embeddings))
    time.sleep(10)  # Wait before the next batch

from langchain.chains.question_answering import load_qa_chain
from langchain.llms import OpenAI

chain= load_qa_chain(OpenAI(), chain_type="stuff")

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAI`.
  warn_deprecated(


In [ ]:
'''
#Initial Query Execution
query="where is the definition to degree of freedom mentioned in the document"
# Assuming you want to search in the first FAISS index:
docs= document_search[0].similarity_search(query)
chain.run(input_documents=docs, question=query)

NameError: name 'document_search' is not defined

In [ ]:
# RAG Implementation
def rag_query(query, max_docs=3): # Limit the number of documents to retrieve
    all_docs = []
    for ds in document_search:
        all_docs.extend(ds.similarity_search(query, k=max_docs)) # Retrieve up to max_docs

    # Generate a response using the retrieved documents
    response = chain.run(input_documents=all_docs, question=query)
    return response

# Example query using RAG, limiting the number of documents
rag_response = rag_query("what is RFID", max_docs=2)
print( rag_response)

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


 RFID stands for Radio Frequency Identification and is a technology used for tracking and identifying objects using radio waves. It involves attaching a small tag or chip to an object and using a reader to gather information about that object, such as its location, movement, and other data.
